In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type = dbutils.widgets.get("load_type")

In [0]:
bronze_df=spark.read.table(f"adb_caresync.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")


In [0]:
from pyspark.sql import functions as F
updated_df=bronze_df.withColumn("insert_timestamp", F.current_timestamp()).withColumn("update_timestamp", F.current_timestamp())

In [0]:
from delta.tables import DeltaTable

target_table = f"adb_caresync.silver.{table_name}"
load_type_norm = load_type.strip().upper()

if load_type_norm == "FULL":
    updated_df.write.mode("overwrite").saveAsTable(target_table)

elif load_type_norm == "APPEND":
    updated_df.write.mode("append").saveAsTable(target_table)

elif load_type_norm == "MERGE":
    keys = [k.strip() for k in merge_keys.split(",") if k.strip()]
    if not keys:
        raise ValueError("merge_keys must be provided (comma-separated) when load_type is MERGE")

    if not spark.catalog.tableExists(target_table):
        # First run: target doesn't exist yet, so create it from the incoming data
        updated_df.write.mode("overwrite").saveAsTable(target_table)
    else:
        # Deduplicate source on merge keys to avoid multiple-match merge errors
        source_df = updated_df.dropDuplicates(keys)
        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in keys])

        # On update, keep the original insert_timestamp and refresh everything else
        update_cols = {f"`{c}`": f"s.`{c}`" for c in source_df.columns if c != "insert_timestamp"}
        insert_cols = {f"`{c}`": f"s.`{c}`" for c in source_df.columns}

        (DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_cols)
            .whenNotMatchedInsert(values=insert_cols)
            .execute())

else:
    raise ValueError(f"Unsupported load_type '{load_type}'. Expected FULL, APPEND or MERGE.")

In [0]:
%sql
select * from adb_caresync.silver.hospitals

In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))